# Day 7 · 图像预处理全链路

**配套讲义**: `days/day-07.md` ｜ **本地可跑**（numpy + Pillow，无 GPU）

处理链：EXIF → 透明底 → resize → 归一化 → pHash。

## 1. 单图处理链可视化

In [ ]:
import sys; sys.path.insert(0, "..")
from PIL import Image, ImageOps
from src.data.image_utils import load_and_normalize

# 造一张「脏图」：EXIF 旋转 + 透明通道
img = Image.new("RGBA", (640, 400), (0, 0, 0, 0))
from PIL import ImageDraw
d = ImageDraw.Draw(img); d.ellipse([100, 50, 500, 350], fill=(200, 90, 60, 255))
img.save("/tmp/_dirty.png", exif=b"") # 实际照片的 EXIF 旋转更常见

out = load_and_normalize("/tmp/_dirty.png")
print("mode:", out.mode, " size:", out.size)
out

## 2. pHash 鲁棒性：噪声 / 缩放 / JPEG 压缩

In [ ]:
import io, random
from src.data.image_utils import phash, hamming_distance

base = load_and_normalize("/tmp/_dirty.png")
h0 = phash(base)

def variant(im, mode):
    if mode == "noise":
        from PIL import ImageFilter
        return im.filter(ImageFilter.GaussianBlur(1))
    if mode == "resize":
        return im.resize((im.width//2, im.height//2)).resize(im.size)
    if mode == "jpeg":
        buf = io.BytesIO(); im.save(buf, "JPEG", quality=40); buf.seek(0)
        return Image.open(buf).convert(im.mode)

for mode in ["noise", "resize", "jpeg"]:
    h = phash(variant(base, mode))
    print(f"{mode:6s} 汉明距离 = {hamming_distance(h0, h):2d}   （≤8 视为重复）")

## 3. 重复图分组（union-find）

In [ ]:
from src.data.image_utils import find_duplicate_groups

paths = ["/tmp/_dirty.png"] * 3 + ["/tmp/_other.png"]
Image.new("RGB", (300, 300), (30, 120, 200)).save("/tmp/_other.png")
groups = find_duplicate_groups(paths, threshold=8)
print("重复组:", groups)

## 4. 作业：对你的商品图批量体检

把 5–200 张图放进 `data/raw_images/`，跑下面的格子。
输出尺寸 / 宽高比 / visual token 分布 —— 这是今天的主产出。

In [ ]:
from pathlib import Path
from src.minivlm.processor import compute_visual_tokens

folder = Path("../data/raw_images")
files = sorted(folder.glob("*")) if folder.exists() else []
if not files:
    print("把图片放进 data/raw_images/ 后重跑。现在用演示图代替。")
    files = ["/tmp/_dirty.png", "/tmp/_other.png"]

rows = []
for f in files:
    try:
        im = load_and_normalize(str(f))
        rows.append((f.name, im.size, round(im.width/im.height, 2),
                     compute_visual_tokens(im.width, im.height)))
    except Exception as e:
        rows.append((f.name, "ERROR", str(e)[:40], "-"))

print(f"{'文件':<24}{'尺寸':<14}{'比例':<7}{'visual tokens'}")
for r in rows:
    print(f"{str(r[0]):<24}{str(r[1]):<14}{str(r[2]):<7}{r[3]}")

## 5. 验收
- [ ] 能讲清「为什么不直接 resize 到 448×448」
- [ ] 批量体检表已产出
- [ ] pHash 阈值 ≤8 的理由能自圆其说